# Ensemble + model comparison (no GPU needed)

Run this after the experiments are finished. It reads everything from MLflow/DagsHub:
1. **Ensemble**: downloads each model's saved logits, averages them, evaluates the result exactly like a single model, and logs it as a new run.
2. **Comparison**: builds the leaderboard (models ranked by **validation** AUROC, never test), a per-label heatmap, validation curves and an accuracy-vs-size chart, logged as a `model-comparison` run.

Settings: **Accelerator None**, Internet ON, the 3 MLflow secrets attached. Works locally too: set the 3 environment variables and run the same commands.


In [ ]:
REPO_URL = "https://github.com/<your-username>/MedVision.git"
ENSEMBLE = ["densenet121_320", "convnext_tiny_320"]     # experiments to combine

In [ ]:
!git clone -q $REPO_URL MedVision
%cd MedVision
!pip install -q tabulate mlflow

In [ ]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    os.environ[k] = s.get_secret(k)          # fails loudly if a secret is missing / not attached
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
!python scripts/ensemble.py --experiments {" ".join(ENSEMBLE)}

In [ ]:
!python scripts/compare_runs.py
from IPython.display import Image, Markdown, display
C = "/kaggle/working/comparison"
display(Markdown(open(f"{C}/leaderboard.md").read()))
for f in ["leaderboard", "per_label_auroc", "val_curves", "auroc_vs_size"]:
    display(Image(f"{C}/{f}.png"))